In [16]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
class LaingRandomForest:
    """
    Random Forest implementation based on Laing et al. 2012 methodology
    for predicting RNA junction structures.
    
    """
    
    def __init__(self, n_estimators=200, max_features='sqrt', 
                 criterion='gini', random_state=42):

        self.n_estimators = n_estimators
        self.max_features = max_features
        self.criterion = criterion
        self.random_state = random_state
        self.rf = None
        self.cv_scores = []
        self.mean_accuracy = 0.0
        self.std_accuracy = 0.0
        self.mean_precision = 0.0
        self.std_precision = 0.0
        self.mean_recall = 0.0
        self.std_recall = 0.0
        
    def cross_validate_laing_method(self, X, y, n_cv_repeats=75, cv_folds=10):

        print(f"Starting Laing 2012 cross-validation method:")
        print(f"- {n_cv_repeats} repetitions of {cv_folds}-fold CV")
        print(f"- {self.n_estimators} trees per random forest")
        print(f"- Features per split: {self.max_features}")
        
        all_accuracy = []
        all_precision = []
        all_recall = []
        all_balanced_accuracy = []
        all_f1 = []
        all_roc = []

        
        for repeat in range(n_cv_repeats):
            # Create stratified k-fold with different random state each time
            skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, 
                                random_state=self.random_state + repeat)
            
            # Create random forest for this repetition
            rf = RandomForestClassifier(
                n_estimators=self.n_estimators,
                max_features=self.max_features,
                criterion=self.criterion,
                random_state=self.random_state + repeat,
                n_jobs=-1
            )
            
            # Perform cross-validation
            cv_accuracy = cross_val_score(rf, X, y, cv=skf, scoring='accuracy')
            all_accuracy.extend(cv_accuracy)
            cv_precision = cross_val_score(rf, X, y, cv=skf, scoring='precision_weighted')
            all_precision.extend(cv_precision)
            cv_recall = cross_val_score(rf, X, y, cv=skf, scoring='recall_weighted')
            all_recall.extend(cv_recall)

            cv_balanced_accuracy = cross_val_score(rf, X, y, cv=skf, scoring='balanced_accuracy')
            all_balanced_accuracy.extend(cv_balanced_accuracy)
            cv_f1 = cross_val_score(rf, X, y, cv=skf, scoring='f1_weighted')
            all_f1.extend(cv_f1)
            y_proba = cross_val_predict(rf, X, y, cv=skf, method="predict_proba")
            roc = roc_auc_score(y, y_proba, multi_class="ovr", average="weighted")
            all_roc.append(roc)

            # Progress indicator
            if (repeat + 1) % 10 == 0:
                current_mean = np.mean(all_accuracy)
                print(f"Completed {repeat + 1}/{n_cv_repeats} repetitions. "
                      f"Current mean accuracy: {current_mean:.4f}")
        
        
        self.cv_scores = all_accuracy
        self.mean_accuracy = np.mean(all_accuracy)
        self.std_accuracy = np.std(all_accuracy)
        self.mean_precision = np.mean(all_precision)
        self.std_precision = np.std(all_precision)
        self.mean_recall = np.mean(all_recall)
        self.std_recall = np.std(all_recall)
        self.mean_balanced_accuracy = np.mean(all_balanced_accuracy)
        self.std_balanced_accuracy = np.std(all_balanced_accuracy)
        self.mean_f1 = np.mean(all_f1)
        self.std_f1 = np.std(all_f1)
        self.mean_roc = np.mean(all_roc)
        self.std_roc = np.std(all_roc)
        results = {
            'all_scores': all_accuracy,
            'mean_accuracy': self.mean_accuracy,
            'std_accuracy': self.std_accuracy,
            'mean_precision': self.mean_precision,
            'std_precision': self.std_precision,
            'mean_recall': self.mean_recall,
            'std_recall': self.std_recall,
            'mean_balanced_accuracy': self.mean_balanced_accuracy,
            'std_balanced_accuracy': self.std_balanced_accuracy,
            'mean_f1': self.mean_f1,
            'std_f1': self.std_f1,
            'mean_roc': self.mean_roc,
            'std_roc': self.std_roc,
            'n_scores': len(all_accuracy),
            'total_experiments': n_cv_repeats * cv_folds
        }
        
        print(f"\nCross-validation completed:")
        print(f"- Total experiments: {results['total_experiments']}")
        print(f"- Mean accuracy: {self.mean_accuracy:.4f} ± {self.std_accuracy:.4f}")
        print(f"- Mean precision: {self.mean_precision:.4f} ± {self.std_precision:.4f}")
        print(f"- Mean recall: {self.mean_recall:.4f} ± {self.std_recall:.4f}")
        print(f"- Min accuracy: {np.min(all_accuracy):.4f}")
        print(f"- Max accuracy: {np.max(all_accuracy):.4f}")
        print(f"- Mean balanced accuracy: {self.mean_balanced_accuracy:.4f} ± {self.std_balanced_accuracy:.4f}")
        print(f"- Mean F1 score: {self.mean_f1:.4f} ± {self.std_f1:.4f}")
        print(f"- Mean ROC AUC: {self.mean_roc:.4f} ± {self.std_roc:.4f}")
        print(f"- Number of scores: {len(all_accuracy)}")
        
        return results
    
    def fit_final_model(self, X, y):

        self.rf = RandomForestClassifier(
            n_estimators=self.n_estimators,
            max_features=self.max_features,
            criterion=self.criterion,
            random_state=self.random_state,
            n_jobs=-1
        )
        
        self.rf.fit(X, y)
        print(f"Final model fitted with {self.n_estimators} trees")
        
    def predict(self, X):

        if self.rf is None:
            raise ValueError("Model not fitted. Call fit_final_model() first.")
        
        return self.rf.predict(X)
    
    def predict_proba(self, X):

        if self.rf is None:
            raise ValueError("Model not fitted. Call fit_final_model() first.")
        
        return self.rf.predict_proba(X)
    
    def get_feature_importance(self):
 
        if self.rf is None:
            raise ValueError("Model not fitted. Call fit_final_model() first.")
        
        return self.rf.feature_importances_



def laing_rf():
    """
    Example implementation following Laing et al. 2012 approach
    """
    print("=== Laing et al. 2012 Random Forest Implementation ===\n")

    X = pd.read_csv('X.csv')
    X.drop(columns=['counts'], inplace=True)
    X = X.values
    y = pd.read_csv('Y.csv').values.ravel() 
    # X = X_4wj.values
    # y = y_4wj.values.ravel()
    

    
    print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features, {len(np.unique(y))} classes")
    
    # Initialize Laing random forest
    laing_rf = LaingRandomForest(
        n_estimators=200,  
        max_features='sqrt',  
        criterion='gini', 
        random_state=42
    )
    

    cv_results = laing_rf.cross_validate_laing_method(
        X, y, 
        n_cv_repeats=1, 
        cv_folds=10
    )
    

    laing_rf.fit_final_model(X, y)
    
   
    feature_importance = laing_rf.get_feature_importance()
    
    print(f"\nFeature Importance Analysis:")
    for i, importance in enumerate(feature_importance):
        print(f"Feature {i+1}: {importance:.4f}")
    
  
    predictions = laing_rf.predict(X[:5])
    probabilities = laing_rf.predict_proba(X[:5])
    
    print(f"\nExample predictions:")
    for i in range(5):
        print(f"Sample {i+1}: Predicted class = {predictions[i]}, "
              f"Probabilities = {probabilities[i]}")
    
    return laing_rf, cv_results




if __name__ == "__main__":
    # Run the example
    laing_rf, cv_results = laing_rf()


=== Laing et al. 2012 Random Forest Implementation ===

Dataset: 112 samples, 15 features, 3 classes
Starting Laing 2012 cross-validation method:
- 1 repetitions of 10-fold CV
- 200 trees per random forest
- Features per split: sqrt

Cross-validation completed:
- Total experiments: 10
- Mean accuracy: 0.8659 ± 0.0900
- Mean precision: 0.8382 ± 0.1175
- Mean recall: 0.8659 ± 0.0900
- Min accuracy: 0.7273
- Max accuracy: 1.0000
- Mean balanced accuracy: 0.8017 ± 0.1248
- Mean F1 score: 0.8421 ± 0.1025
- Mean ROC AUC: 0.9465 ± 0.0000
- Number of scores: 10
Final model fitted with 200 trees

Feature Importance Analysis:
Feature 1: 0.0645
Feature 2: 0.0982
Feature 3: 0.0703
Feature 4: 0.0941
Feature 5: 0.0808
Feature 6: 0.0802
Feature 7: 0.0375
Feature 8: 0.0273
Feature 9: 0.0424
Feature 10: 0.0563
Feature 11: 0.0594
Feature 12: 0.0610
Feature 13: 0.0458
Feature 14: 0.0556
Feature 15: 0.1266

Example predictions:
Sample 1: Predicted class = 2, Probabilities = [0.015 0.005 0.98 ]
Sample 2: P